# 🇪🇬 Fine-Tuning Qwen2.5-7B for Egyptian Document OCR (QLoRA + Unsloth)
This notebook fine-tunes **Qwen2.5-7B-Instruct** on Egyptian government administrative documents (National IDs, Birth Certificates, Licenses, Utility Bills) to repair noisy OCR and output clean structured JSON.

**Hardware:** Runs on **Free Google Colab T4 GPU** (~15-25 minutes).
**Output:** Exports a **Q4_K_M GGUF** model file to run 100% locally in Ollama on a 4GB VRAM PC.

### 1. Install Unsloth & Dependencies

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

### 2. Load Base Model: Qwen2.5-7B-Instruct in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### 3. Load & Format Dataset
Upload `egypt_docs_train.jsonl` from your `sahelha-backend/backend/data/` folder into Colab files.

In [ ]:
from datasets import load_dataset

# Load your 1,080 Egyptian document pairs
dataset = load_dataset("json", data_files="egypt_docs_train.jsonl", split="train")

def format_prompts(examples):
    texts = []
    for messages in examples["messages"]:
        # Use Qwen chat template
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print(f"Loaded {len(dataset)} examples successfully!")

### 4. Train with SFTTrainer (~20 mins on Free Colab T4)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 200,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

### 5. Export Directly to Q4_K_M GGUF
Unsloth quantizes and creates the GGUF file directly in Colab.

In [ ]:
# Export to 4-bit Quantized GGUF (Q4_K_M)
model.save_pretrained_gguf("qwen2.5-7b-egypt-ocr", tokenizer, quantization_method = "q4_k_m")

print("GGUF Export Complete! Look in your Colab files for: qwen2.5-7b-egypt-ocr-Q4_K_M.gguf")

### 6. Run in Local Ollama on your PC
Once downloaded to your PC, create a `Modelfile`:
```dockerfile
FROM ./qwen2.5-7b-egypt-ocr-Q4_K_M.gguf
```
And run:
```bash
ollama create egypt-ocr-7b -f Modelfile
ollama run egypt-ocr-7b
```